In [ ]:
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd

file_path = '/Users/natemekonen/Desktop/Data_Projects/nba_2for1_analysis/data'

# Pull regular season and playoffs game ids
regular_df = leaguegamelog.LeagueGameLog(
    season = '2025-26',
    season_type_all_star = 'Regular Season',
    player_or_team_abbreviation = 'T'
).get_data_frames()[0]

playoffs_df = leaguegamelog.LeagueGameLog(
    season = '2025-26',
    season_type_all_star = 'Playoffs',
    player_or_team_abbreviation = 'T'
).get_data_frames()[0]

all_season = pd.concat([regular_df.assign(SEASON_TYPE="Regular Season"), playoffs_df.assign(SEASON_TYPE="Playoffs")], ignore_index=True)

game_ids = all_season['GAME_ID'].unique()

print(len(game_ids))


1315


In [ ]:
from nba_api.stats.endpoints import playbyplayv3
import pandas as pd
import time
import pyarrow

# Pull play by play data
all_pbp = []

for game_id in game_ids:
    try:
            pbp = playbyplayv3.PlayByPlayV3(
                game_id = game_id,
                start_period = 1,
                end_period = 10 
            )

            plays = pbp.get_data_frames()[0]
            plays['GAME_ID'] = game_id

            all_pbp.append(plays)

            print(f'Completed {game_id}')

            time.sleep(0.6)

    except Exception as e:
        print(f'Failed {game_id}: {e}')

# Combine all play dataframes into one 
pbp_df = pd.concat(all_pbp, ignore_index=True)

pbp_df.to_parquet(f'{file_path}/nba_2025_26_play_by_play.parquet', index=False)

Completed 0022500001
Completed 0022500002
Completed 0022500089
Completed 0022500080
Completed 0022500003
Completed 0022500081
Completed 0022500083
Completed 0022500088
Completed 0022500004
Completed 0022500082
Completed 0022500087
Completed 0022500085
Completed 0022500084
Completed 0022500086
Completed 0022500005
Completed 0022500006
Completed 0022500019
Completed 0022500091
Completed 0022500018
Completed 0022500090
Completed 0022500099
Completed 0022500097
Completed 0022500098
Completed 0022500095
Completed 0022500092
Completed 0022500096
Completed 0022500093
Completed 0022500094
Completed 0022500100
Completed 0022500103
Completed 0022500102
Completed 0022500104
Completed 0022500101
Completed 0022500110
Completed 0022500105
Completed 0022500108
Completed 0022500112
Completed 0022500113
Completed 0022500111
Completed 0022500106
Completed 0022500107
Completed 0022500109
Completed 0022500008
Completed 0022500116
Completed 0022500114
Completed 0022500120
Completed 0022500122
Completed 002

In [29]:
# Read in parquet file with play by play data
pbp_plays = pd.read_parquet(f'{file_path}/nba_2025_26_play_by_play.parquet')

pd.set_option('display.max_columns', None)

# Sort the df in chronological order
pbp_plays = pbp_plays.sort_values(
    ['GAME_ID', 'period', 'actionNumber']
)

pbp_plays.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,GAME_ID
0,0022500001,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (7:44 PM EST),period,start,1,0,1,0022500001
1,0022500001,4,PT12M00.00S,1,1610612760,OKC,1631096,Holmgren,C. Holmgren,0,0,0,,0,,,0,h,Jump Ball Holmgren vs. Adams: Tip to Thompson,Jump Ball,,1,0,2,0022500001
2,0022500001,8,PT11M36.00S,1,1610612745,HOU,1630578,Sengun,A. Sengun,-179,179,25,Missed,1,,,0,v,MISS Sengun 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,3,0022500001
3,0022500001,9,PT11M31.00S,1,1610612760,OKC,1641717,Wallace,C. Wallace,0,0,0,,0,,,0,h,Wallace REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,4,0022500001
4,0022500001,10,PT11M26.00S,1,1610612760,OKC,1631096,Holmgren,C. Holmgren,0,13,1,Made,1,2,0,2,h,Holmgren 1' Cutting Layup Shot (2 PTS) (Dort 1...,Made Shot,Cutting Layup Shot,1,2,5,0022500001


In [30]:
# Define function to convert clock columns into seconds
def clock_to_seconds(clock):

    if pd.isna(clock):
        return None

    clock = str(clock)

    clock = (
        clock
        .replace('PT', '')
        .replace('S', '')
    )

    minutes, seconds = clock.split('M')

    return int(minutes) * 60 + float(seconds)

In [ ]:
# Define function to parse data and create possessions
import re

def create_possessions(game_df):
    
    possessions = []
    possession_id = 1
    current_events = []
    possession_team = None
    start_clock = None
    free_throw_total = None
    free_throw_number = None

    def save_possession():

        nonlocal possession_id
        nonlocal current_events
        nonlocal start_clock
        nonlocal possession_team

        if len(current_events) == 0:
            return

        # Remove blocks from output
        output_events = [
            e for e in current_events
            if not (
                (pd.isna(e['actionType']) or e['actionType'] == '')
                and pd.notna(e.get('description'))
                and 'BLOCK' in str(e['description'])
            )
        ]

        # Get the first shot in the possession
        first_shot = next(
            (e for e in output_events if e.get('actionType') in ['Made Shot', 'Missed Shot']),
            None
        )

        if first_shot is not None:
            first_shot_seconds = round(clock_to_seconds(first_shot.get('clock')), 0)
            first_shot_player = first_shot.get('playerNameI')
            first_shot_result = first_shot.get('shotResult')
            first_shot_value = first_shot.get('shotValue')
            first_shot_x_legacy = first_shot.get('xLegacy')
            first_shot_y_legacy = first_shot.get('yLegacy')
            first_shot_distance = first_shot.get('shotDistance')
            first_shot_subtype = first_shot.get('subType')

        else:
            first_shot_seconds = None
            first_shot_player = None
            first_shot_result = None
            first_shot_value = None
            first_shot_x_legacy = None
            first_shot_y_legacy = None
            first_shot_distance = None
            first_shot_subtype = None

        possessions.append({
            'GAME_ID': current_events[0]['GAME_ID'],
            'POSSESSION_ID': possession_id,
            'PERIOD': current_events[0]['period'],
            'TEAM': current_events[-1].get('teamTricode'),
            'START_CLOCK': start_clock,
            'END_CLOCK': current_events[-1]['clock'],
            'EVENTS': len(output_events),
            'ACTION_TYPES':
                ', '.join(
                    [
                        (
                            'Steal'
                            if ((pd.isna(e['actionType']) or e['actionType'] == '') and 'STEAL' in str(e.get('description')))
                            else str(e['actionType'])
                        )
                        for e in output_events
                    ]
                ),
            'CLOCK_LIST': ', '.join([str(e['clock']) for e in output_events]),
            'SHOT_VALUE': next(
                (
                    e['shotValue']
                    for e in reversed(output_events)
                    if pd.notna(e.get('shotValue'))
                ),
                None
            ),
            'TEAM_ID': current_events[-1].get('teamId'),
            'PLAYER_ID': current_events[-1].get('personId'),
            'PLAYER_NAME': current_events[-1].get('playerNameI'),
            'HOME_SCORE': current_events[-1].get('scoreHome'),
            'AWAY_SCORE': current_events[-1].get('scoreAway'),
            'X_LEGACY': current_events[-1].get('xLegacy'),
            'Y_LEGACY': current_events[-1].get('yLegacy'),
            'SHOT_DISTANCE': current_events[-1].get('shotDistance'),
            'SHOT_RESULT': current_events[-1].get('shotResult'),
            'SUBTYPE': current_events[-1].get('subType'),
            'FIRST_SHOT_SECONDS': first_shot_seconds,
            'FIRST_SHOT_PLAYER': first_shot_player,
            'FIRST_SHOT_RESULT': first_shot_result,
            'FIRST_SHOT_VALUE': first_shot_value,
            'FIRST_SHOT_X_LEGACY': first_shot_x_legacy,
            'FIRST_SHOT_Y_LEGACY': first_shot_y_legacy,
            'FIRST_SHOT_DISTANCE': first_shot_distance,
            'FIRST_SHOT_SUBTYPE': first_shot_subtype,
            'EVENT_LIST': ' | '.join([str(e['description']) for e in output_events])
        })

        possession_id += 1
        current_events = []
        start_clock = None
        possession_team = None

    for _, row in game_df.iterrows():

        action = row['actionType']

        if action == 'Substitution':
            continue

        # Convert row to dictionary
        event = row.to_dict()

        # Ignore quarter start messages
        if action == 'period' and len(current_events) == 0:
            continue

        # Start possession
        if len(current_events) == 0:
            start_clock = row['clock']

            # team responsible for event
            if pd.notna(row['teamTricode']):
                possession_team = row['teamTricode']

        current_events.append(event)

        # Track shots
        if action in ['Made Shot', 'Missed Shot']:
            last_shot_team = row['teamTricode']

        # Track free throws
        if action == 'Free Throw':
            match = re.search(
                r'Free Throw (\d+) of (\d+)',
                str(row['description'])
            )
            if match:
                free_throw_number = int(match.group(1))
                free_throw_total = int(match.group(2))

        possession_end = False

        # Made basket ends possession
        if action == 'Made Shot':
            possession_end = True

        # Turnover ends possession
        elif action == 'Turnover':
            possession_end = True

        # Missed shot followed by defensive rebound
        # Missed shot followed by block + defensive rebound
        if action == 'Missed Shot':
            current_idx = game_df.index.get_loc(row.name)
            future_events = game_df.iloc[current_idx + 1:]

            for _, next_row in future_events.iterrows():
                # Defensive rebound ends possession
                if next_row['actionType'] == 'Rebound':
                    rebound_team = next_row['teamTricode']

                    if (
                        pd.notna(rebound_team)
                        and rebound_team != row['teamTricode']
                        and rebound_team != 'Unknown'
                    ):
                        possession_end = True

                    break

                # Ignore blocks, steals, and blank action types
                if (
                    (pd.isna(next_row['actionType']) or next_row['actionType'] == '')
                    and (
                        'BLOCK' in str(next_row['description'])
                        or 'STEAL' in str(next_row['description'])
                    )
                ):
                    continue

                # Stop searching if another shot happens
                if next_row['actionType'] in [
                    'Made Shot',
                    'Missed Shot'
                ]:
                    break

        # Final free throw
        elif action == 'Free Throw':
            if (
                free_throw_number is not None
                and free_throw_total is not None
                and free_throw_number == free_throw_total
            ):
                possession_end = True

        # End of quarter
        elif action == 'period':
            possession_end = True

        if possession_end:
            save_possession()
            free_throw_number = None
            free_throw_total = None

    return possessions

In [158]:
import numpy as np

# Group pbp data into games and create possessions for each
all_possessions = []

for game_id, game in pbp_plays.groupby('GAME_ID'):

    game = (
        game
        .sort_values(['period', 'actionNumber'])
        .copy()
    )

    game['scoreHome'] = (
        game['scoreHome']
        .replace('', np.nan)
        .ffill()
        .fillna(0)
    )

    game['scoreAway'] = (
        game['scoreAway']
        .replace('', np.nan)
        .ffill()
        .fillna(0)
    )

    game_possessions = create_possessions(game)

    all_possessions.extend(game_possessions)

In [159]:
import re

# Define function to get free throw points
def get_free_throw_points(event_list):
    
    if pd.isna(event_list):
        return 0
    
    points = 0
    
    for event in event_list.split(' | '):
        
        if 'Free Throw' in event:
            
            # Missed free throw
            if 'MISS' in event.upper():
                continue
            
            # Made free throw
            points += 1
    
    return points

In [ ]:
import numpy as np

possession_df = pd.DataFrame(all_possessions)

# Convert clock columns to seconds
possession_df['START_SECONDS'] = (
    possession_df['START_CLOCK']
    .apply(clock_to_seconds)
    .round(0)
    .astype(int)
)

possession_df['END_SECONDS'] = (
    possession_df['END_CLOCK']
    .apply(clock_to_seconds)
    .round(0)
    .astype(int)
)

possession_df['CLOCK_SECONDS_LIST'] = (
    possession_df['CLOCK_LIST']
    .str.split(',')
    .apply(
        lambda clocks: ', '.join(
            str(round(clock_to_seconds(c.strip())))
            for c in clocks
        )
    )
)

# Convert scores to numeric
possession_df['HOME_SCORE'] = pd.to_numeric(
    possession_df['HOME_SCORE'],
    errors='coerce'
)

possession_df['AWAY_SCORE'] = pd.to_numeric(
    possession_df['AWAY_SCORE'],
    errors='coerce'
)

# Get the final action in the possession
last_action = possession_df['ACTION_TYPES'].str.split(',').str[-1].str.strip()

# Get the first action in the possession
first_action = possession_df['ACTION_TYPES'].str.split(',').str[0].str.strip()

# Calculate possession length
possession_df['POSS_LENGTH'] = np.where(
    last_action.isin(['Made Shot', 'Missed Shot']),
    possession_df['START_SECONDS'] - possession_df['END_SECONDS'],
    np.nan
)

# Handle possession length where the shot is the only action
only_shot = (possession_df['EVENTS'] == 1) & (last_action.isin(['Made Shot', 'Missed Shot']))

possession_df.loc[only_shot, 'POSS_LENGTH'] = (possession_df['END_SECONDS'].shift(1) - possession_df['START_SECONDS'])

# Handle Missed Shot -> other event -> Made/Missed Shot
three_event_shot = (
    (possession_df['EVENTS'] == 3)
    &
    (first_action == 'Missed Shot')
    &
    (last_action.isin(['Made Shot', 'Missed Shot']))
)

possession_df.loc[three_event_shot, 'POSS_LENGTH'] = (
    possession_df['END_SECONDS'].shift(1)
    -
    possession_df['END_SECONDS']
)

# Find how many seconds passed before the first shot in the possession
possession_df['TIME_UNTIL_FIRST_SHOT'] = round(
    possession_df['START_SECONDS'] - possession_df['FIRST_SHOT_SECONDS'],
    2
)

# Create official possession time
possession_df['SHOT_TIME_FOR_ANALYSIS'] = np.where(
    possession_df['EVENTS'] <= 2,
    possession_df['POSS_LENGTH'],
    np.where(
        first_action == 'Missed Shot',
        possession_df['POSS_LENGTH'],
        possession_df['TIME_UNTIL_FIRST_SHOT']
    )
)

# Check if the possession is a 2-for-1 attempt
possession_df['TWO_FOR_ONE_ATTEMPT'] = np.where(
    (
        possession_df['START_SECONDS'].between(34, 40)
        &
        (possession_df['SHOT_TIME_FOR_ANALYSIS'] <= 7)
        &
        ((possession_df['PERIOD'] != 4) |
            ((possession_df['PERIOD'] == 4) & ((possession_df['HOME_SCORE'] - possession_df['AWAY_SCORE']).abs() <= 10))
        )
    ),
    1,
    0
)

# Check if the possession is not a 2-for-1 attempt
possession_df['NO_TWO_FOR_ONE_ATTEMPT'] = np.where(
    (
        possession_df['POSS_LENGTH'].notna()
        &
        possession_df['START_SECONDS'].between(34, 40)
        &
        (possession_df['SHOT_TIME_FOR_ANALYSIS'] > 7)
        &
        (possession_df['END_SECONDS'] <= 24)
        & 
        ((possession_df['PERIOD'] != 4) | 
                ((possession_df['PERIOD'] == 4) & ((possession_df['HOME_SCORE'] - possession_df['AWAY_SCORE']).abs() <= 10))
        )
    ),
    1,
    0
)

# Create lookup of matchups
team_lookup = (
    possession_df.groupby('GAME_ID')['TEAM']
    .unique()
    .apply(list)
)

# Create opponent column
possession_df['OPPONENT'] = possession_df.apply(
    lambda x: team_lookup[x['GAME_ID']][
        1 - team_lookup[x['GAME_ID']].index(x['TEAM'])
    ],
    axis=1
)

# Calculate free throw points
possession_df['FREE_THROW_PTS'] = (
    possession_df['EVENT_LIST']
    .apply(get_free_throw_points)
)

# Create points on possession column
possession_df['POSS_PTS'] = np.where(
    possession_df['FREE_THROW_PTS'] > 0,
    possession_df['FREE_THROW_PTS'],
    np.where(
        possession_df['SHOT_RESULT'] == 'Made',
        possession_df['SHOT_VALUE'],
        np.where(
            possession_df['SHOT_RESULT'] == 'Missed',
            0,
            np.nan
        )
    )
)

possession_df['POSS_PTS'] = possession_df['POSS_PTS'].fillna(0)

possession_df.head()

,GAME_ID,POSSESSION_ID,PERIOD,TEAM,START_CLOCK,END_CLOCK,EVENTS,ACTION_TYPES,CLOCK_LIST,SHOT_VALUE,TEAM_ID,PLAYER_ID,PLAYER_NAME,HOME_SCORE,AWAY_SCORE,X_LEGACY,Y_LEGACY,SHOT_DISTANCE,SHOT_RESULT,SUBTYPE,FIRST_SHOT_SECONDS,FIRST_SHOT_PLAYER,FIRST_SHOT_RESULT,FIRST_SHOT_VALUE,FIRST_SHOT_X_LEGACY,FIRST_SHOT_Y_LEGACY,FIRST_SHOT_DISTANCE,FIRST_SHOT_SUBTYPE,EVENT_LIST,START_SECONDS,END_SECONDS,CLOCK_SECONDS_LIST,POSS_LENGTH,TIME_UNTIL_FIRST_SHOT,SHOT_TIME_FOR_ANALYSIS,TWO_FOR_ONE_ATTEMPT,NO_TWO_FOR_ONE_ATTEMPT,OPPONENT,FREE_THROW_PTS,POSS_PTS
0,0022500001,1,1,HOU,PT12M00.00S,PT11M36.00S,2,"Jump Ball, Missed Shot","PT12M00.00S, PT11M36.00S",3,1610612745,1630578,A. Sengun,0,0,-179,179,25,Missed,Jump Shot,696.0,A. Sengun,Missed,3.0,-179.0,179.0,25.0,Jump Shot,Jump Ball Holmgren vs. Adams: Tip to Thompson ...,720,696,"720, 696",24.0,24.0,24.0,0,0,OKC,0,0.0
1,0022500001,2,1,OKC,PT11M31.00S,PT11M26.00S,2,"Rebound, Made Shot","PT11M31.00S, PT11M26.00S",2,1610612760,1631096,C. Holmgren,2,0,0,13,1,Made,Cutting Layup Shot,686.0,C. Holmgren,Made,2.0,0.0,13.0,1.0,Cutting Layup Shot,Wallace REBOUND (Off:0 Def:1) | Holmgren 1' Cu...,691,686,"691, 686",5.0,5.0,5.0,0,0,HOU,0,2.0
2,0022500001,3,1,OKC,PT11M26.00S,PT11M26.00S,2,"Foul, Free Throw","PT11M26.00S, PT11M26.00S",0,1610612760,1631096,C. Holmgren,3,0,0,0,0,,Free Throw 1 of 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Smith Jr. S.FOUL (P1.T1) (E.Dalen) | Holmgren ...,686,686,"686, 686",NaN,NaN,NaN,0,0,HOU,1,1.0
3,0022500001,4,1,HOU,PT11M08.00S,PT11M02.00S,3,"Missed Shot, Rebound, Made Shot","PT11M08.00S, PT11M04.00S, PT11M02.00S",2,1610612745,1631095,J. Smith Jr.,3,2,-7,-5,1,Made,Putback Layup Shot,668.0,A. Thompson,Missed,3.0,233.0,0.0,0.0,Jump Shot,MISS Thompson 3PT Jump Shot | Smith Jr. REBOUN...,668,662,"668, 664, 662",24.0,0.0,24.0,0,0,OKC,0,2.0
4,0022500001,5,1,OKC,PT10M50.00S,PT10M50.00S,1,Made Shot,PT10M50.00S,2,1610612760,1631096,C. Holmgren,5,2,64,39,7,Made,Floating Jump shot,650.0,C. Holmgren,Made,2.0,64.0,39.0,7.0,Floating Jump shot,Holmgren 7' Floating Jump Shot (5 PTS) (Wallac...,650,650,650,12.0,0.0,12.0,0,0,HOU,0,2.0


In [ ]:
# Sort the full possession dataframe chronologically
possession_df = possession_df.sort_values(
    ['GAME_ID', 'PERIOD', 'POSSESSION_ID']
).reset_index(drop=True)

# Number of later possessions for the same team in the same quarter
possession_df['LATER_TEAM_POSSESSIONS'] = (
    possession_df
    .groupby(['GAME_ID', 'PERIOD', 'TEAM'])
    .cumcount(ascending=False)
)

# Did the team get another possession later in the quarter after a two-for-one?
possession_df['TWO_FOR_ONE_SECOND_POSSESSION?'] = np.where(
    (
        (possession_df['TWO_FOR_ONE_ATTEMPT'] == 1)
        &
        (possession_df['LATER_TEAM_POSSESSIONS'] > 0)
    ),
    1,
    np.where(
        possession_df['TWO_FOR_ONE_ATTEMPT'] == 1,
        0,
        np.nan
    )
)

# Did the team get another possession later in the quarter after running the clock?
possession_df['NON_TWO_FOR_ONE_SECOND_POSSESSION?'] = np.where(
    (
        (possession_df['NO_TWO_FOR_ONE_ATTEMPT'] == 1)
        &
        (possession_df['LATER_TEAM_POSSESSIONS'] > 0)
    ),
    1,
    np.where(
        possession_df['NO_TWO_FOR_ONE_ATTEMPT'] == 1,
        0,
        np.nan
    )
)

# Get the points scored on possession after two-for-one
def next_team_possession_points(group):

    next_pts = []

    for i in range(len(group)):

        if i < len(group) - 1:
            next_pts.append(
                group.iloc[i + 1]['POSS_PTS']
            )
        else:
            next_pts.append(np.nan)

    return pd.Series(
        next_pts,
        index=group.index
    )

# Create column that shows pts on the possession after two-for-one
possession_df['SECOND_POSSESSION_PTS'] = (
    possession_df
    .groupby(
        ['GAME_ID', 'PERIOD', 'TEAM'],
        group_keys=False
    )
    .apply(next_team_possession_points)
)

# Get first shot result from the team's next possession
def next_team_possession_shot_result(group):

    next_result = []

    for i in range(len(group)):

        if i < len(group) - 1:
            next_result.append(
                group.iloc[i + 1]['FIRST_SHOT_RESULT']
            )
        else:
            next_result.append(np.nan)

    return pd.Series(
        next_result,
        index=group.index
    )

# Get last shot result from the team's next possession
def next_team_possession_shot_result(group):

    next_result = []

    for i in range(len(group)):

        if i < len(group) - 1:
            next_result.append(
                group.iloc[i + 1]['SHOT_RESULT']
            )
        else:
            next_result.append(np.nan)

    return pd.Series(
        next_result,
        index=group.index
    )

possession_df['SECOND_POSSESSION_SHOT_RESULT'] = (
    possession_df
    .groupby(
        ['GAME_ID', 'PERIOD', 'TEAM'],
        group_keys=False
    )
    .apply(next_team_possession_shot_result)
)

# Only show the second possession for two-for-ones or non-two-for-ones
possession_df['SECOND_POSSESSION_SHOT_RESULT'] = np.where(
    (
        (possession_df['TWO_FOR_ONE_SECOND_POSSESSION?'] == 1)
        |
        (possession_df['NON_TWO_FOR_ONE_SECOND_POSSESSION?'] == 1)
    ),
    possession_df['SECOND_POSSESSION_SHOT_RESULT'],
    np.nan
)

# Get the first shot time from the team's next possession
def next_team_possession_shot_seconds(group):

    next_shot_seconds = []

    for i in range(len(group)):

        if i < len(group) - 1:
            next_shot_seconds.append(
                group.iloc[i + 1]['FIRST_SHOT_SECONDS']
            )
        else:
            next_shot_seconds.append(np.nan)

    return pd.Series(
        next_shot_seconds,
        index=group.index
    )


possession_df['SECOND_POSSESSION_SHOT_SECONDS'] = (
    possession_df
    .groupby(
        ['GAME_ID', 'PERIOD', 'TEAM'],
        group_keys=False
    )
    .apply(next_team_possession_shot_seconds)
)

possession_df['SECOND_POSSESSION_SHOT_SECONDS'] = np.where(
    (
        (possession_df['TWO_FOR_ONE_SECOND_POSSESSION?'] == 1)
        |
        (possession_df['NON_TWO_FOR_ONE_SECOND_POSSESSION?'] == 1)
    ),
    possession_df['SECOND_POSSESSION_SHOT_SECONDS'],
    np.nan
)

# Get points for second possession
possession_df['TWO_FOR_ONE_SECOND_POSSESSION_PTS'] = np.where(
    possession_df['TWO_FOR_ONE_SECOND_POSSESSION?'] == 1,
    possession_df['SECOND_POSSESSION_PTS'],
    np.where(
        possession_df['NON_TWO_FOR_ONE_SECOND_POSSESSION?'] == 1,
        possession_df['SECOND_POSSESSION_PTS'],
        0
    )
)

possession_df['TOTAL_PTS_ON_ALL_2FOR1_POSS'] = possession_df['POSS_PTS'] + possession_df['TWO_FOR_ONE_SECOND_POSSESSION_PTS']

# Filter df for possessions in the final minute where possession length and shot time are not negative
final_minute_df = possession_df[
    (possession_df['START_SECONDS'] <= 60)
    &
    (possession_df['POSS_LENGTH'] > 0)
    &
    (possession_df['SHOT_TIME_FOR_ANALYSIS'] > 0)
]

final_minute_df = final_minute_df[['GAME_ID', 'PERIOD', 'POSSESSION_ID', 'TEAM', 'OPPONENT', 'X_LEGACY', 'Y_LEGACY', 'SHOT_DISTANCE', 'SUBTYPE', 'FIRST_SHOT_PLAYER', 'FIRST_SHOT_RESULT', 
                                   'FIRST_SHOT_VALUE', 'SHOT_TIME_FOR_ANALYSIS', 'CLOCK_SECONDS_LIST', 'TWO_FOR_ONE_ATTEMPT', 'NO_TWO_FOR_ONE_ATTEMPT', 
                                   'POSS_PTS', 'EVENT_LIST', 'NON_TWO_FOR_ONE_SECOND_POSSESSION?', 'TWO_FOR_ONE_SECOND_POSSESSION?', 'TWO_FOR_ONE_SECOND_POSSESSION_PTS', 'TOTAL_PTS_ON_ALL_2FOR1_POSS']]

In [ ]:
import re
import unicodedata

# Get player and team logo images

team_logos = pd.read_csv(f'{file_path}/team_logos.csv', index_col=0)
player_images = pd.read_csv(f'{file_path}/player_images.csv', index_col=0)

def normalize_player_name_for_matching(name):

    if pd.isna(name):
        return name

    # Remove accents
    name = ''.join(
        c for c in unicodedata.normalize('NFD', str(name))
        if unicodedata.category(c) != 'Mn'
    )

    # Remove suffixes
    name = re.sub(
        r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$',
        '',
        name,
        flags=re.IGNORECASE
    )

    # Remove extra whitespace
    name = re.sub(r'\s+', ' ', name).strip()

    # Split name
    parts = name.split()

    if len(parts) < 2:
        return name.lower()

    # First initial + last name
    return f'{parts[0][0]}. {parts[-1]}'.lower()


team_abbrev_map = {
    'Atlanta Hawks': 'ATL',
    'Boston Celtics': 'BOS',
    'Brooklyn Nets': 'BKN',
    'Charlotte Hornets': 'CHA',
    'Chicago Bulls': 'CHI',
    'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL',
    'Denver Nuggets': 'DEN',
    'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW',
    'Houston Rockets': 'HOU',
    'Indiana Pacers': 'IND',
    'LA Clippers': 'LAC',
    'Los Angeles Lakers': 'LAL',
    'Memphis Grizzlies': 'MEM',
    'Miami Heat': 'MIA',
    'Milwaukee Bucks': 'MIL',
    'Minnesota Timberwolves': 'MIN',
    'New Orleans Pelicans': 'NOP',
    'New York Knicks': 'NYK',
    'Oklahoma City Thunder': 'OKC',
    'Orlando Magic': 'ORL',
    'Philadelphia 76ers': 'PHI',
    'Phoenix Suns': 'PHX',
    'Portland Trail Blazers': 'POR',
    'Sacramento Kings': 'SAC',
    'San Antonio Spurs': 'SAS',
    'Toronto Raptors': 'TOR',
    'Utah Jazz': 'UTA',
    'Washington Wizards': 'WAS'
}

team_logos['TEAM_ABBREV'] = (
    team_logos['team']
    .map(team_abbrev_map)
)

player_images['Player_Normalized'] = (
    player_images['Player']
    .apply(normalize_player_name_for_matching)
)

team_logos = team_logos[['team', 'TEAM_ABBREV', 'logo']]
player_images = player_images[player_images['Team'] != '2TM']
player_images["Team"] = player_images["Team"].replace({
    'PHO': 'PHX',
    'CHO': 'CHA',
    'BRK': 'BKN'
})
player_images = player_images[['Player','Player_Normalized', 'Team', 'Images']]

In [166]:
# Create final dataframe with player and team images

filtered_df = possession_df[
    (possession_df['TWO_FOR_ONE_ATTEMPT'] == 1)
    |
    (possession_df['NO_TWO_FOR_ONE_ATTEMPT'] == 1)
].copy()

filtered_df = filtered_df[filtered_df['TIME_UNTIL_FIRST_SHOT'] >= 0]

filtered_df['PLAYER_NAME_NORMALIZED'] = (
    filtered_df['FIRST_SHOT_PLAYER']
    .apply(normalize_player_name_for_matching)
)

pbp_logos_df = filtered_df.merge(team_logos, how='left', left_on='TEAM', right_on='TEAM_ABBREV')
final_df = pbp_logos_df.merge(player_images, how='left', left_on=['PLAYER_NAME_NORMALIZED', 'TEAM'], right_on=['Player_Normalized', 'Team'])
final_df = final_df.drop_duplicates(subset=['GAME_ID', 'PERIOD', 'POSSESSION_ID', 'PLAYER_NAME'])
keep_columns = ['GAME_ID','POSSESSION_ID','PERIOD','START_CLOCK','END_CLOCK','EVENTS','ACTION_TYPES','EVENT_LIST','TEAM','team','TEAM_ID','PLAYER_ID','PLAYER_NAME',
                'HOME_SCORE','AWAY_SCORE','FIRST_SHOT_RESULT','FIRST_SHOT_VALUE','FIRST_SHOT_SUBTYPE','FIRST_SHOT_PLAYER','FIRST_SHOT_X_LEGACY','FIRST_SHOT_Y_LEGACY',
                'FIRST_SHOT_DISTANCE','FIRST_SHOT_SECONDS','POSS_LENGTH','TIME_UNTIL_FIRST_SHOT','SHOT_TIME_FOR_ANALYSIS','TWO_FOR_ONE_ATTEMPT',
                'NO_TWO_FOR_ONE_ATTEMPT','POSS_PTS','FREE_THROW_PTS','TWO_FOR_ONE_SECOND_POSSESSION?','NON_TWO_FOR_ONE_SECOND_POSSESSION?','SECOND_POSSESSION_PTS',
                'SECOND_POSSESSION_SHOT_RESULT','SECOND_POSSESSION_SHOT_SECONDS','TWO_FOR_ONE_SECOND_POSSESSION_PTS','TOTAL_PTS_ON_ALL_2FOR1_POSS','TEAM_ABBREV',
                'logo','Player','Images']

final_df = final_df[keep_columns]

final_df = final_df[
    ~(
        (final_df["EVENTS"] == 1) &
        ((final_df["FIRST_SHOT_SECONDS"] + final_df["POSS_LENGTH"]) > 40)
    )
]

final_df.to_csv(
    f'{file_path}/final_2for1_stats.csv',
    index=False
)

In [168]:
two_for_one_shots = (final_df['TWO_FOR_ONE_ATTEMPT'] == 1).sum()
non_two_for_one_shots = (final_df['NO_TWO_FOR_ONE_ATTEMPT'] == 1).sum()

print(f'Two for One Shot Attempts: {two_for_one_shots}')
print(f'Non Two for One Shot Attempts: {non_two_for_one_shots}')

Two for One Shot Attempts: 582
Non Two for One Shot Attempts: 196


In [169]:
import pandas as pd
import numpy as np

# Keep only rows with a valid first-shot distance
shot_df = final_df[
    final_df["FIRST_SHOT_DISTANCE"].notna()
].copy()

# Create distance buckets
bins = [0, 5, 10, 15, 20, 25, np.inf]
labels = [
    "0-5 feet",
    "5-10 feet",
    "10-15 feet",
    "15-20 feet",
    "20-25 feet",
    "25+ feet"
]

shot_df["DISTANCE_BUCKET"] = pd.cut(
    shot_df["FIRST_SHOT_DISTANCE"],
    bins=bins,
    labels=labels,
    right=False  # 0-5 includes 0 through <5
)

# 2-for-1 distribution
two_dist = (
    shot_df[shot_df["TWO_FOR_ONE_ATTEMPT"] == 1]
    .groupby("DISTANCE_BUCKET", observed=False)
    .size()
    .pipe(lambda x: x / x.sum() * 100)
    .round(1)
)

# Non-2-for-1 distribution
non_dist = (
    shot_df[shot_df["NO_TWO_FOR_ONE_ATTEMPT"] == 1]
    .groupby("DISTANCE_BUCKET", observed=False)
    .size()
    .pipe(lambda x: x / x.sum() * 100)
    .round(1)
)

# Combine into one table
distance_table = pd.DataFrame({
    "2-for-1 %": two_dist,
    "Non-2-for-1 %": non_dist
}).fillna(0)

print(distance_table)

                 2-for-1 %  Non-2-for-1 %
DISTANCE_BUCKET                          
0-5 feet              39.9           29.1
5-10 feet             11.0           14.3
10-15 feet             4.0            6.1
15-20 feet             2.9            7.7
20-25 feet             5.3            6.6
25+ feet              36.9           36.2
